#### DDL

In [0]:
-- create schema
CREATE SCHEMA IF NOT EXISTS PO_MART

In [0]:
-- DIM_DATE
CREATE TABLE IF NOT EXISTS PO_MART.DIM_DATE
(
    Date_ID BIGINT NOT NULL,
    INPUT_DATE DATE NOT NULL,
    INPUT_YEAR INT NOT NULL,
    INPUT_MONTH INT NOT NULL,
    INPUT_DAY INT NOT NULL
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_DATE"
;

In [0]:
-- DIM_DOCUMENT

CREATE TABLE IF NOT EXISTS PO_MART.DIM_DOC
(
    Doc_ID VARCHAR(25) NOT NULL,
    DOCUMENT_NUMBER INT NOT NULL,
    RECORD_TYPE CHAR(2) NOT NULL,
    SOURCE_TYPE CHAR(2) NOT NULL,
    DESCRIPTION VARCHAR(500),
    TYPE_CODE VARCHAR(10),
    TYPE_DESCRIPTION VARCHAR(20),
    STATUS_CODE INT,
    STATUS_DESCRIPTION VARCHAR(500)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_DOC"
;

In [0]:
-- DIM_PO_CATEGORY

CREATE TABLE IF NOT EXISTS PO_MART.DIM_PO_CATEGORY
(
    ID BIGINT NOT NULL,
    CODE VARCHAR(255),
    DESCRIPTION VARCHAR(500)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_PO_CATEGORY"
;

In [0]:
-- DIM_DEPARTMENT
CREATE TABLE IF NOT EXISTS PO_MART.DIM_DEPT
(
    ID BIGINT NOT NULL,
    NUMBER INT,
    NAME VARCHAR(100)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_DEPT"
;

In [0]:
-- DIM_COST_CENTER
CREATE TABLE IF NOT EXISTS PO_MART.DIM_COST_CENTER
(
    ID BIGINT NOT NULL,
    COST_CENTER INT,
    NAME VARCHAR(100)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_COST_CENTER"
;

In [0]:
-- DIM_EMPLOYEE
CREATE TABLE IF NOT EXISTS PO_MART.DIM_EMPLOYEE
(
    ID BIGINT NOT NULL,
    INPUT_BY VARCHAR(100),
    PURCHASING_AGENT VARCHAR(100)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_EMPLOYEE"
;

In [0]:
-- DIM_VENDOR
CREATE TABLE IF NOT EXISTS PO_MART.DIM_VENDOR
(
    ID BIGINT NOT NULL,
    NUMBER INT,
    NAME_1 VARCHAR(255),
    NAME_2 VARCHAR(255),
    TYPE VARCHAR(50),
    ADDRESS_1 VARCHAR(255),
    ADDRESS_2 VARCHAR(255),
    CITY VARCHAR(100),
    STATE VARCHAR(10),
    ZIP VARCHAR(20),
    STATUS VARCHAR(2),
    CLASS VARCHAR(10),
    CONTACT_NAME VARCHAR(255),
    CONTACT_TITLE VARCHAR(100),
    CONTACT_PHONE BIGINT,
    CONTACT_EXTENSION INT,
    GENDER VARCHAR(100),
    ETHNICITY VARCHAR(100),
    MINORITY VARCHAR(100),
    MINORITY_DESCRIPTION VARCHAR(100),
    DISADVANTAGED BOOLEAN,
    DISABLED_VETERAN BOOLEAN,
    SB_DISABLED_VET BOOLEAN,
    SB_MINORITY BOOLEAN,
    SB_MINORITY_WOMAN BOOLEAN,
    SB_NON_MINORITY BOOLEAN,
    SB_DISADVANTAGED BOOLEAN,
    SB_VETERAN BOOLEAN,
    SB_WOMAN BOOLEAN,
    GEOGRAPHIC_AREA VARCHAR(10),
    INDEPENDENT_CONTRACTOR BOOLEAN
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_VENDOR"
;

In [0]:
-- DIM_ITEM
CREATE TABLE IF NOT EXISTS PO_MART.DIM_ITEM
(
    ID BIGINT NOT NULL,
    NUMBER INT,
    DESCRIPTION STRING,
    UNIT_OF_MEASURE VARCHAR(20),
    UNIT_OF_MEASURE_DESCRIPTION VARCHAR(100)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_ITEM"
;

In [0]:
-- DIM_COMMODITY
CREATE TABLE IF NOT EXISTS PO_MART.DIM_COMMODITY
(
    ID BIGINT NOT NULL,
    CODE INT,
    DESCRIPTION VARCHAR(255)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_COMMODITY"
;

In [0]:
-- DIM_EXPENSE_TYPE
CREATE TABLE IF NOT EXISTS PO_MART.DIM_EXPENSE_TYPE
(
    ID BIGINT NOT NULL,
    TYPE INT,
    DESCRIPTION VARCHAR(255)
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/DIM_EXPENSE_TYPE"
;

In [0]:
-- FACT_PO_ORDER_ITEM
CREATE TABLE IF NOT EXISTS PO_MART.FACT_PO_ORDER_ITEM
(
    Date_ID BIGINT NOT NULL,
    Doc_ID VARCHAR(25) NOT NULL,
    ID BIGINT NOT NULL,
    START_DATE DATE,
    EXPIRATION_DATE DATE,
    EXTENSION_DATE DATE,
    Item_Number INT,
    Requisition_Number INT,
    Unique_ID VARCHAR(25) NOT NULL,
    Total_Items FLOAT,
    Item_Quantity_Ordered INT,
    Item_Unit_Cost FLOAT,
    Item_Total_Cost FLOAT,
    Total_Amount FLOAT,
    Vouched_Amount FLOAT,
    PO_Balance FLOAT,
    Annual_Contract INT,
    _INGESTED_AT_ VARCHAR(20) NOT NULL,
    __FILE_SOURCE__ VARCHAR(20) NOT NULL
)
USING DELTA
LOCATION "s3://purchase-orders-aws/po-mart/FACT_PO_ORDER_ITEM"
;

#### Procedures

In [0]:
-- dim_commodity
MERGE INTO PO_MART.dim_commodity AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                COMMODITY_CODE,
                COMMODITY_DESCRIPTION
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID

        WHEN MATCHED THEN
        UPDATE SET
            target.CODE = source.COMMODITY_CODE,
            target.DESCRIPTION = source.COMMODITY_DESCRIPTION
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            CODE,
            DESCRIPTION
        )
        VALUES
        (
            source.ID,
            source.COMMODITY_CODE,
            source.COMMODITY_DESCRIPTION
        );

In [0]:
-- dim_cost_center
MERGE INTO PO_MART.dim_cost_center AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                COST_CENTER,
                COST_CENTER_NAME
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID
        
        WHEN MATCHED THEN
        UPDATE SET
            target.COST_CENTER = source.COST_CENTER,
            target.NAME = source.COST_CENTER_NAME
        
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            COST_CENTER,
            NAME
        )
        VALUES
        (
            source.ID,
            source.COST_CENTER,
            source.COST_CENTER_NAME
        );

In [0]:
-- dim_date
MERGE INTO PO_MART.dim_date AS target
USING (
    WITH max_date AS (
        SELECT COALESCE(MAX(Date_ID), 0) AS max_date_id
        FROM PO_MART.dim_date
    ),
    new_dates AS (
        SELECT DISTINCT
            INPUT_DATE
        FROM purchaseorders
        WHERE INPUT_DATE IS NOT NULL
    )
    SELECT
        ROW_NUMBER() OVER (ORDER BY n.INPUT_DATE)
            + (SELECT max_date_id FROM max_date) AS Date_ID,
        n.INPUT_DATE,
        YEAR(n.INPUT_DATE)  AS INPUT_YEAR,
        MONTH(n.INPUT_DATE) AS INPUT_MONTH,
        DAY(n.INPUT_DATE)   AS INPUT_DAY
    FROM new_dates n
) AS source
ON target.INPUT_DATE = source.INPUT_DATE

WHEN NOT MATCHED THEN
INSERT (
    Date_ID,
    INPUT_DATE,
    INPUT_YEAR,
    INPUT_MONTH,
    INPUT_DAY
)
VALUES (
    source.Date_ID,
    source.INPUT_DATE,
    source.INPUT_YEAR,
    source.INPUT_MONTH,
    source.INPUT_DAY
);



In [0]:
-- dim_dept
MERGE INTO PO_MART.dim_dept AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                DEPARTMENT_NUMBER,
                DEPARTMENT_NAME
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID

        WHEN MATCHED THEN
        UPDATE SET
            target.NUMBER = source.DEPARTMENT_NUMBER,
            target.NAME = source.DEPARTMENT_NAME
        
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            NUMBER,
            NAME
        )
        VALUES
        (
            source.ID,
            source.DEPARTMENT_NUMBER,
            source.DEPARTMENT_NAME
        );

In [0]:
-- dim_doc
WITH cte_ranked AS
(
    SELECT
        DOCUMENT_NUMBER,
        RECORD_TYPE,
        SOURCE_DOCUMENT_TYPE,
        DOCUMENT_DESCRIPTION,
        DOCUMENT_TYPE_CODE,
        DOCUMENT_TYPE_DESCRIPTION,
        DOCUMENT_STATUS_CODE,
        DOCUMENT_STATUS_DESCRIPTION,
        ROW_NUMBER() OVER
        (
            PARTITION BY DOCUMENT_NUMBER, INPUT_DATE
            ORDER BY ITEM_NUMBER DESC
        ) AS rn
    FROM purchaseorders
    WHERE DOCUMENT_NUMBER IS NOT NULL
),
cte_doc_id AS
(
    SELECT
        CONCAT(DOCUMENT_NUMBER, '_', rn) AS Doc_ID,
        DOCUMENT_NUMBER,
        RECORD_TYPE,
        SOURCE_DOCUMENT_TYPE,
        DOCUMENT_DESCRIPTION,
        DOCUMENT_TYPE_CODE,
        DOCUMENT_TYPE_DESCRIPTION,
        DOCUMENT_STATUS_CODE,
        DOCUMENT_STATUS_DESCRIPTION,
        ROW_NUMBER() OVER
        (
            PARTITION BY CONCAT(DOCUMENT_NUMBER, '_', rn)
            ORDER BY DOCUMENT_NUMBER
        ) AS dedup_rn
    FROM cte_ranked
),
cte_dedup AS
(
    SELECT *
    FROM cte_doc_id
    WHERE dedup_rn = 1   -- 🔑 THIS LINE FIXES THE ERROR
)

MERGE INTO PO_MART.dim_doc AS target
USING cte_dedup AS source
ON target.Doc_ID = source.Doc_ID

WHEN MATCHED THEN
    UPDATE SET
        target.RECORD_TYPE        = source.RECORD_TYPE,
        target.SOURCE_TYPE        = source.SOURCE_DOCUMENT_TYPE,
        target.DESCRIPTION        = source.DOCUMENT_DESCRIPTION,
        target.TYPE_CODE          = source.DOCUMENT_TYPE_CODE,
        target.TYPE_DESCRIPTION   = source.DOCUMENT_TYPE_DESCRIPTION,
        target.STATUS_CODE        = source.DOCUMENT_STATUS_CODE,
        target.STATUS_DESCRIPTION = source.DOCUMENT_STATUS_DESCRIPTION

WHEN NOT MATCHED THEN
    INSERT
    (
        Doc_ID,
        DOCUMENT_NUMBER,
        RECORD_TYPE,
        SOURCE_TYPE,
        DESCRIPTION,
        TYPE_CODE,
        TYPE_DESCRIPTION,
        STATUS_CODE,
        STATUS_DESCRIPTION
    )
    VALUES
    (
        source.Doc_ID,
        source.DOCUMENT_NUMBER,
        source.RECORD_TYPE,
        source.SOURCE_DOCUMENT_TYPE,
        source.DOCUMENT_DESCRIPTION,
        source.DOCUMENT_TYPE_CODE,
        source.DOCUMENT_TYPE_DESCRIPTION,
        source.DOCUMENT_STATUS_CODE,
        source.DOCUMENT_STATUS_DESCRIPTION
    );


In [0]:
-- dim_employee
MERGE INTO PO_MART.dim_employee AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                INPUT_BY,
                PURCHASING_AGENT
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID

        WHEN MATCHED THEN
        UPDATE SET
            target.INPUT_BY = source.INPUT_BY,
            target.PURCHASING_AGENT = source.PURCHASING_AGENT
        
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            INPUT_BY,
            PURCHASING_AGENT
        )
        VALUES
        (
            source.ID,
            source.INPUT_BY,
            source.PURCHASING_AGENT
        );

In [0]:
-- dim_expense_type
MERGE INTO PO_MART.dim_expense_type AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                EXPENSE_TYPE,
                EXPENSE_TYPE_DESCRIPTION
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID

        WHEN MATCHED THEN
        UPDATE SET
            target.TYPE = source.EXPENSE_TYPE,
            target.DESCRIPTION = source.EXPENSE_TYPE_DESCRIPTION
        
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            TYPE,
            DESCRIPTION
        )
        VALUES
        (
            source.ID,
            source.EXPENSE_TYPE,
            source.EXPENSE_TYPE_DESCRIPTION
        );

In [0]:
-- dim_item
MERGE INTO PO_MART.dim_item AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                ITEM_NUMBER,
                ITEM_DESCRIPTION,
                ITEM_UNIT_OF_MEASURE,
                ITEM_UNIT_OF_MEASURE_DESCRIPTION
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID

        WHEN MATCHED THEN
        UPDATE SET
            target.NUMBER = source.ITEM_NUMBER,
            target.DESCRIPTION = source.ITEM_DESCRIPTION,
            target.UNIT_OF_MEASURE = source.ITEM_UNIT_OF_MEASURE,
            target.UNIT_OF_MEASURE_DESCRIPTION = source.ITEM_UNIT_OF_MEASURE_DESCRIPTION
        
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            NUMBER,
            DESCRIPTION,
            UNIT_OF_MEASURE,
            UNIT_OF_MEASURE_DESCRIPTION
        )
        VALUES
        (
            source.ID,
            source.ITEM_NUMBER,
            source.ITEM_DESCRIPTION,
            source.ITEM_UNIT_OF_MEASURE,
            source.ITEM_UNIT_OF_MEASURE_DESCRIPTION
        );

In [0]:
-- dim_po_category
MERGE INTO PO_MART.dim_po_category AS target 
        USING
        (
            SELECT 
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                PO_CATEGORY_CODE,
                PO_CATEGORY_DESCRIPTION
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
        ) AS source
        ON target.ID = source.ID

        WHEN MATCHED THEN
        UPDATE SET
            target.CODE = source.PO_CATEGORY_CODE,
            target.DESCRIPTION = source.PO_CATEGORY_DESCRIPTION
        
        WHEN NOT MATCHED BY TARGET THEN
        INSERT
        (
            ID,
            CODE,
            DESCRIPTION
        )
        VALUES
        (
            source.ID,
            source.PO_CATEGORY_CODE,
            source.PO_CATEGORY_DESCRIPTION
        );

In [0]:
-- dim_vendor
MERGE INTO PO_MART.dim_vendor AS target
        USING
        (
            SELECT
                RIGHT(UNIQUE_ID, LEN(UNIQUE_ID) - 2) AS ID,
                VENDOR_NUMBER,
                VENDOR_NAME_1,
                VENDOR_NAME_2,
                VENDOR_ADDRESS_1,
                VENDOR_ADDRESS_2,
                VENDOR_CITY,
                VENDOR_STATE,
                VENDOR_ZIP,
                VENDOR_CONTACT_NAME,
                VENDOR_CONTACT_TITLE,
                VENDOR_CONTACT_PHONE,
                VENDOR_CONTACT_EXTENSION,
                VENDOR_TYPE,
                GENDER,
                ETHNICITY,
                STATUS,
                CLASS,
                GEOGRAPHIC_AREA,
                INDEPENDENT_CONTRACTOR,
                MINORITY,
                VENDOR_MINORITY_DESCRIPTION,
                DISADVANTAGED,
                DISABLED_VETERAN,
                SB_DISABLED_VET,
                SB_MINORITY,
                SB_MINORITY_WOMAN,
                SB_NON_MINORITY,
                SB_DISADVANTAGED,
                SB_VETERAN,
                SB_WOMAN
            FROM purchaseorders
            WHERE UNIQUE_ID IS NOT NULL
              AND VENDOR_NAME_1 IS NOT NULL
        ) AS source
        ON target.ID = source.ID
        WHEN MATCHED THEN
            UPDATE SET
                target.NUMBER         = source.VENDOR_NUMBER,
                target.NAME_1                = source.VENDOR_NAME_1,
                target.NAME_2                = source.VENDOR_NAME_2,
                target.ADDRESS_1             = source.VENDOR_ADDRESS_1,
                target.ADDRESS_2             = source.VENDOR_ADDRESS_2,
                target.CITY                  = source.VENDOR_CITY,
                target.STATE                 = source.VENDOR_STATE,
                target.ZIP                   = source.VENDOR_ZIP,
                target.STATUS                = source.STATUS,
                target.CLASS                 = source.CLASS,
                target.CONTACT_NAME          = source.VENDOR_CONTACT_NAME,
                target.CONTACT_TITLE         = source.VENDOR_CONTACT_TITLE,
                target.CONTACT_PHONE         = source.VENDOR_CONTACT_PHONE,
                target.CONTACT_EXTENSION     = source.VENDOR_CONTACT_EXTENSION,
                target.GENDER                = source.GENDER,
                target.ETHNICITY             = source.ETHNICITY,
                target.MINORITY              = source.MINORITY,
                target.MINORITY_DESCRIPTION  = source.VENDOR_MINORITY_DESCRIPTION,
                target.DISADVANTAGED          = CAST(source.DISADVANTAGED AS BOOLEAN),
                target.DISABLED_VETERAN       = CAST(source.DISABLED_VETERAN AS BOOLEAN),
                target.SB_DISABLED_VET        = CAST(source.SB_DISABLED_VET AS BOOLEAN),
                target.SB_MINORITY            = CAST(source.SB_MINORITY AS BOOLEAN),
                target.SB_MINORITY_WOMAN      = CAST(source.SB_MINORITY_WOMAN AS BOOLEAN),
                target.SB_NON_MINORITY        = CAST(source.SB_NON_MINORITY AS BOOLEAN),
                target.SB_DISADVANTAGED       = CAST(source.SB_DISADVANTAGED AS BOOLEAN),
                target.SB_VETERAN             = CAST(source.SB_VETERAN AS BOOLEAN),
                target.SB_WOMAN               = CAST(source.SB_WOMAN AS BOOLEAN),
                target.GEOGRAPHIC_AREA        = source.GEOGRAPHIC_AREA,
                target.INDEPENDENT_CONTRACTOR = CAST(source.INDEPENDENT_CONTRACTOR AS BOOLEAN)
        WHEN NOT MATCHED BY TARGET THEN
            INSERT
            (
                ID,
                NUMBER,
                NAME_1,
                NAME_2,
                TYPE,
                ADDRESS_1,
                ADDRESS_2,
                CITY,
                STATE,
                ZIP,
                STATUS,
                CLASS,
                CONTACT_NAME,
                CONTACT_TITLE,
                CONTACT_PHONE,
                CONTACT_EXTENSION,
                GENDER,
                ETHNICITY,
                MINORITY,
                MINORITY_DESCRIPTION,
                DISADVANTAGED,
                DISABLED_VETERAN,
                SB_DISABLED_VET,
                SB_MINORITY,
                SB_MINORITY_WOMAN,
                SB_NON_MINORITY,
                SB_DISADVANTAGED,
                SB_VETERAN,
                SB_WOMAN,
                GEOGRAPHIC_AREA,
                INDEPENDENT_CONTRACTOR
            )
            VALUES
            (
                source.ID,
                source.VENDOR_NUMBER,
                source.VENDOR_NAME_1,
                source.VENDOR_NAME_2,
                source.VENDOR_TYPE,
                source.VENDOR_ADDRESS_1,
                source.VENDOR_ADDRESS_2,
                source.VENDOR_CITY,
                source.VENDOR_STATE,
                source.VENDOR_ZIP,
                source.STATUS,
                source.CLASS,
                source.VENDOR_CONTACT_NAME,
                source.VENDOR_CONTACT_TITLE,
                source.VENDOR_CONTACT_PHONE,
                source.VENDOR_CONTACT_EXTENSION,
                source.GENDER,
                source.ETHNICITY,
                source.MINORITY,
                source.VENDOR_MINORITY_DESCRIPTION,
                CAST(source.DISADVANTAGED AS BOOLEAN),
                CAST(source.DISABLED_VETERAN AS BOOLEAN),
                CAST(source.SB_DISABLED_VET AS BOOLEAN),
                CAST(source.SB_MINORITY AS BOOLEAN),
                CAST(source.SB_MINORITY_WOMAN AS BOOLEAN),
                CAST(source.SB_NON_MINORITY AS BOOLEAN),
                CAST(source.SB_DISADVANTAGED AS BOOLEAN),
                CAST(source.SB_VETERAN AS BOOLEAN),
                CAST(source.SB_WOMAN AS BOOLEAN),
                source.GEOGRAPHIC_AREA,
                CAST(source.INDEPENDENT_CONTRACTOR AS BOOLEAN)
            );

In [0]:
-- fact_po_order_item
WITH doc_rn AS
(
    SELECT
        DOCUMENT_NUMBER,
        INPUT_DATE,
        ITEM_NUMBER,
        ROW_NUMBER() OVER
        (
            PARTITION BY DOCUMENT_NUMBER, INPUT_DATE
            ORDER BY ITEM_NUMBER DESC
        ) AS rn
    FROM purchaseorders
    WHERE DOCUMENT_NUMBER IS NOT NULL
)
INSERT INTO PO_MART.fact_po_order_item
(
    Date_ID,
    Doc_ID,
    ID,
    START_DATE,
    EXPIRATION_DATE,
    EXTENSION_DATE,
    Item_Number,
    Requisition_Number,
    Unique_ID,
    Total_Items,
    Item_Quantity_Ordered,
    Item_Unit_Cost,
    Item_Total_Cost,
    Total_Amount,
    Vouched_Amount,
    PO_Balance,
    Annual_Contract,
    _INGESTED_AT_,
    __FILE_SOURCE__
)
SELECT
    d.Date_ID,
    CONCAT(p.DOCUMENT_NUMBER, '_', r.rn) AS Doc_ID,  
    RIGHT(p.UNIQUE_ID, LEN(p.UNIQUE_ID) - 2) AS ID,
    p.START_DATE,
    p.EXPIRATION_DATE,
    p.EXTENSION_DATE,
    p.ITEM_NUMBER,
    p.REQUISITION_NUMBER,
    p.UNIQUE_ID,
    p.TOTAL_ITEMS,
    p.ITEM_QUANTITY_ORDERED,
    p.ITEM_UNIT_COST,
    p.ITEM_TOTAL_COST,
    p.TOTAL_AMOUNT,
    p.VOUCHED_AMOUNT,
    p.PO_BALANCE,
    p.ANNUAL_CONTRACT,
    p._INGESTED_AT_,
    p.__FILE_SOURCE__
FROM purchaseorders p
JOIN doc_rn r
    ON  p.DOCUMENT_NUMBER = r.DOCUMENT_NUMBER
    AND p.INPUT_DATE = r.INPUT_DATE
    AND p.ITEM_NUMBER = r.ITEM_NUMBER
JOIN PO_MART.dim_date d
    ON d.INPUT_DATE = p.INPUT_DATE
WHERE p.UNIQUE_ID IS NOT NULL
  AND NOT EXISTS
  (
      SELECT 1
      FROM PO_MART.fact_po_order_item f
      WHERE f.Unique_ID = p.UNIQUE_ID
        AND f.Item_Number = p.ITEM_NUMBER
  );
